# Lab 07 — Annotation Quality and Disagreement

In this lab, we explore how to **evaluate, combine, and improve human annotations** using real crowdsourced datasets.

We begin with the **Recognizing Textual Entailment (RTE)** dataset, which contains multiple judgments per item and reference labels. We will measure agreement and accuracy, investigate disagreement, and compare different aggregation methods. We will also simulate worker qualification, filtering, and continuous quality control to examine their effects on annotation quality and coverage.

Finally, we will briefly explore **subjective emotion ratings**, where disagreement may reflect meaningful differences in human interpretation rather than annotation errors.

## 1. Getting to Know the Data

In this lab, we will investigate the quality of annotations collected from crowdsourcing workers. In this context, *workers* are the human annotators - it is a term that is commonly used in crowdsourcing. 

We will start with the **Recognizing Textual Entailment (RTE)** dataset from Snow et al. (2008).

In this task, workers read a text and decide whether a hypothesis follows from it.

- **1 - Entailed:** The hypothesis follows from the text.
- **0 - Not entailed:** The hypothesis does not follow from the text.

Each item was annotated by multiple workers. The dataset also includes a reference label that we can use later to evaluate their judgments.

Before proceeding, make sure you run the command below from the repository root. This download the dataset from Snow et al. (2008).

```bash
python labs/Lab07/prepare_data.py
```

### 1.1 Load the dataset

Load the CSV with pandas and take a first look at its structure.


In [ ]:
import pandas as pd
from IPython.display import display

rte = pd.read_csv("data/processed/rte.csv")

print(f"Annotations: {len(rte):,}")
print(f"Items: {rte['item_id'].nunique():,}")
print(f"Workers: {rte['worker_id'].nunique():,}")

display(rte[["worker_id", "item_id", "text", "hypothesis", "response", "reference_label"]].head())

### 1.2 One item, multiple judgments

Let's look at a single item where annotators gave different answers.

We will display the original text, the hypothesis, and all the worker responses.

Notice that the dataset contains both **individual human annotations** and an accepted **reference label**.

In [ ]:
# Find one item where workers disagree.
different_labels = rte.groupby("item_id")["response"].nunique()
item_id = different_labels[different_labels > 1].index[0]

item = rte[rte["item_id"] == item_id]

print("Text:", item["text"].iloc[0])
print("\nHypothesis:", item["hypothesis"].iloc[0])
print("\nReference label:", item["reference_label"].iloc[0])

display(item[["worker_id", "response"]])

### 1.3 How was the annotation work distributed?

We have seen that several workers can annotate the same item. But how was the work distributed across the crowd?

Let's examine two things:

- **Annotations per item:** How many people judged each item?
- **Annotations per worker:** How many items did each person annotate?

These distributions matter because not every worker necessarily contributes the same amount of data.

**Questions to consider:**
- Did every item receive the same number of judgments?
- Did all workers contribute equally?
- Why might this matter when evaluating individual annotators?

In [ ]:
import matplotlib.pyplot as plt

# Count annotations per item and per worker.
per_item = rte.groupby("item_id").size()
per_worker = rte.groupby("worker_id").size()

print("Annotations per item:")
print(per_item.describe()[["min", "50%", "max"]])

print("\nAnnotations per worker:")
print(per_worker.describe()[["min", "50%", "max"]])

# Visualize worker contributions.
worker_counts = per_worker.value_counts().sort_index()

plt.figure(figsize=(9, 4))
plt.bar(
    worker_counts.index.astype(str),
    worker_counts.values,
    edgecolor="black"
)

plt.xlabel("Annotations per worker")
plt.ylabel("Number of workers")
plt.title("How much did each worker contribute?")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Measuring Annotation Quality

### 2.1 How much do annotators agree?

We saw that different workers sometimes give different answers to the same item.

Each RTE item has 10 annotations. Let's examine how those judgments are distributed.

For each item, count how many workers selected **Entailed (1)**. The result can range from 0 to 10:

- **0 or 10:** All workers agree.
- **5:** Workers are evenly split.
- **1 or 9:** One worker disagrees with the other nine.

Plot the distribution across all items.

**Questions:**
- Are unanimous judgments common?
- How many items show substantial disagreement?

In [ ]:
# Count votes for each response on every item.
votes = (
    rte.groupby(["item_id", "response"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
)

votes.columns = ["not_entailed", "entailed"]

# Number of workers selecting "Entailed" per item.
entailed_votes = votes["entailed"]

distribution = (
    entailed_votes.value_counts()
    .reindex(range(11), fill_value=0)
    .sort_index()
)

plt.figure(figsize=(8, 4))
plt.bar(distribution.index, distribution.values, edgecolor="black")
plt.xticks(range(11))
plt.xlabel("Number of workers selecting 'Entailed' (out of 10)")
plt.ylabel("Number of items")
plt.title("How are worker judgments distributed?")
plt.tight_layout()
plt.show()

print(f"Unanimous items: {distribution[0] + distribution[10]}")
print(f"Evenly split items (5–5): {distribution[5]}")

### 2.2 Measuring agreement

We can measure agreement by comparing pairs of workers who annotated the same item.

**Raw pairwise agreement** is the proportion of annotation pairs that assigned the same label.

$$
P_{\text{agree}} = \frac{\text{Number of agreeing pairs}}{\text{Total number of annotation pairs}}
$$

However, some agreement can happen simply because workers choose from the same set of categories.

**Krippendorff's alpha (α)** adjusts observed disagreement for the disagreement expected by chance.

$$
\alpha = 1 - \frac{D_o}{D_e}
$$

Where:
- $D_o$ = **Observed disagreement** among annotators.
- $D_e$ = **Expected disagreement** by chance, based on the overall label distribution.

**Interpretation:**
- **α = 1:** Perfect agreement.
- **α = 0:** Agreement no better than expected by chance.
- **α < 0:** Less agreement than expected by chance.

Let's calculate raw pairwise agreement and nominal Krippendorff's alpha for RTE.

**Question:** How does the interpretation change after accounting for chance agreement?

In [ ]:
# Number of possible annotation pairs per item.
n = votes.sum(axis=1)
total_pairs = n * (n - 1) / 2

# Pairs that assigned the same response.
agreeing_pairs = (votes * (votes - 1) / 2).sum(axis=1)

# Raw pairwise agreement across all items.
raw_agreement = agreeing_pairs.sum() / total_pairs.sum()

# Observed disagreement.
observed_disagreement = 1 - raw_agreement

# Expected disagreement from the overall label frequencies.
N = len(rte)
n0 = (rte["response"] == 0).sum()
n1 = (rte["response"] == 1).sum()

expected_disagreement = 2 * n0 * n1 / (N * (N - 1))

# Nominal Krippendorff's alpha.
alpha = 1 - observed_disagreement / expected_disagreement

print(f"Raw pairwise agreement: {raw_agreement:.1%}")
print(f"Krippendorff's alpha: {alpha:.3f}")

### 2.3 Agreement is not accuracy

Workers can agree with one another and still disagree with the reference label.

Our dataset includes an accepted reference label for each item, allowing us to calculate **reference-based accuracy**.

- **Agreement:** How often do workers assign the same label?
- **Accuracy:** How often does a worker's response match the reference label?

Let's calculate overall accuracy and compare agreement with accuracy at the item level.

Remember that a reference label is an accepted benchmark—not necessarily an independently verified fact.

**Question:** Can you find items where workers largely agree with one another but disagree with the reference?


In [ ]:
# Compare each individual response with its reference.
rte["matches_reference"] = (
    rte["response"] == rte["reference_label"]
)

overall_accuracy = rte["matches_reference"].mean()
print(f"Accuracy against reference: {overall_accuracy:.1%}")

In [ ]:
# For each item, count how many workers matched the reference.
correct_votes = (
    rte["response"].eq(rte["reference_label"])
    .groupby(rte["item_id"])
    .sum()
)

distribution = (
    correct_votes.value_counts()
    .reindex(range(11), fill_value=0)
    .sort_index()
)

# Let's plot the distribution of workers per item that had been correct
# notice that when that number > 5, then the majority is correct
plt.figure(figsize=(9, 4))
plt.bar(distribution.index, distribution.values, edgecolor="black")

plt.xticks(range(11))
plt.xlabel("Workers matching reference (out of 10)")
plt.ylabel("Number of items")
plt.title("How often does the crowd match the reference?")
plt.tight_layout()
plt.show()

print("Majority matches reference:", (correct_votes >= 6).sum())
print("Evenly split:", (correct_votes == 5).sum())
print("Majority disagrees with reference:", (correct_votes <= 4).sum())

## 3. Investigating Disagreement

### 3.1 Which items cause disagreement?

Overall agreement tells us how consistent the annotations are, but not **why** workers disagree.

Let's identify an item where workers are divided and examine the original text and hypothesis.

For a binary task with 10 annotations, a 5–5 split indicates the greatest disagreement.

**Questions:**
- Why might workers interpret this item differently?
- Is the text ambiguous, or might the task instructions be unclear?
- What additional information would help resolve the disagreement?

In [ ]:
# Helper to inspect any item.
def inspect_item(item_id):
    item = rte[rte["item_id"] == item_id]

    print("Text:", item["text"].iloc[0])
    print("\nHypothesis:", item["hypothesis"].iloc[0])
    print("\nReference:", item["reference_label"].iloc[0])
    print("\nWorker responses:")
    print(item["response"].value_counts().sort_index())

    
# Count how many workers voted "Entailed" (1) for each item.
item_stats = (
    rte.assign(
        correct=rte["response"] == rte["reference_label"]
    )
    .groupby("item_id")
    .agg(
        total=("response", "size"),
        entailed=("response", "sum"),
        matches_reference=("correct", "sum")
    )
)

# A 5–5 split means maximum disagreement.
item_stats["disagreement"] = (
    item_stats["entailed"] - 5
).abs()

# Sort from most to least disputed (smallest distance first).
disputed_items = item_stats.sort_values("disagreement")

# Inspect the first, second, or third most disputed item.
inspect_item(disputed_items.index[0])

### 3.2 What if the crowd disagrees with the reference?

Some items had a majority of workers disagreeing with the reference label.
Let's examine one of these cases.

We will look for items where **at most 2 of the 10 workers matched the reference**.

**Questions:**
- Does the reference label seem reasonable?
- Can you understand why most workers chose a different answer?
- What would you do with this item before using it for model training?

**Remember:** Strong agreement against a reference does not prove that the reference is wrong. It tells us that the item deserves closer investigation.

In [ ]:
# Find items where most workers disagree with the reference.
candidates = item_stats[
    item_stats["matches_reference"] <= 2
].sort_values("matches_reference")

print("Items with strong crowd-reference disagreement:", len(candidates))
print()

if not candidates.empty:
    item_id = candidates.index[3]
    inspect_item(item_id)


### 3.3 Do all workers perform equally?

Some workers may be more consistent with the reference labels than others.

We can estimate each worker's **reference-based accuracy**: the proportion of their annotations that match the reference.

But remember that workers annotated different numbers of items.

Let's compare each worker's accuracy with the number of annotations they contributed.

**Questions:**
- Do all workers have similar accuracy?
- Would you trust an accuracy estimate based on 20 annotations as much as one based on 800?
- Could differences in accuracy reflect the difficulty of the items assigned to each worker?

In [ ]:
# Calculate accuracy and annotation count for each worker.
worker_stats = (
    rte.assign(
        correct=rte["response"] == rte["reference_label"]
    )
    .groupby("worker_id")
    .agg(
        annotations=("correct", "size"),
        accuracy=("correct", "mean")
    )
)

display(worker_stats.describe().round(2))


# Mean of individual worker accuracies (each worker counts equally).
mean_worker_accuracy = worker_stats["accuracy"].mean()

# Accuracy across all annotations (each annotation counts equally).
overall_accuracy = (
    rte["response"] == rte["reference_label"]
).mean()

plt.figure(figsize=(8, 4))

plt.scatter(
    worker_stats["annotations"],
    worker_stats["accuracy"],
    alpha=0.5
)

plt.axhline(
    mean_worker_accuracy,
    color="green",
    linestyle="--",
    label=f"Mean worker accuracy ({mean_worker_accuracy:.1%})"
)

plt.axhline(
    overall_accuracy,
    color="red",
    linestyle=":",
    label=f"Overall annotation accuracy ({overall_accuracy:.1%})"
)

plt.xlabel("Number of worker annotations")
plt.ylabel("Accuracy against reference")
plt.title("Worker accuracy vs. number of annotations")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Aggregating Annotations

### 4.1 Majority Voting

So far, we have worked with individual annotations. But what if we need **one label per item** to train a machine learning model?

A simple approach is **majority voting**: select the label chosen by most annotators.

For our RTE dataset, each item has 10 judgments:

- **More than 5 vote Entailed (1):** Assign label 1.
- **Fewer than 5 vote Entailed (1):** Assign label 0.
- **Exactly 5 vote Entailed:** Tie — no majority label.

Let's aggregate the annotations and compare the result with the reference labels.

**Questions:**
- How many items have a majority label?
- Does majority voting improve accuracy compared with individual judgments?

In [ ]:
# Count votes and retrieve the reference label for each item.
majority = rte.groupby("item_id").agg(
    entailed_votes=("response", "sum"),
    total_votes=("response", "size"),
    reference_label=("reference_label", "first")
)

# Assign the label selected by the majority.
majority["majority_label"] = (
    majority["entailed_votes"] > majority["total_votes"] / 2
).astype("Int64")

# A 5–5 split has no majority.
ties = majority["entailed_votes"] == majority["total_votes"] / 2
majority.loc[ties, "majority_label"] = pd.NA

print("Sample of items, with their majority label:")
display(majority.head(10))

# Compare individual annotations with majority voting.
individual_accuracy = (
    rte["response"] == rte["reference_label"]
).mean()

# Evaluate majority voting.
decided = majority.dropna(subset=["majority_label"])

correct = (
    decided["majority_label"] == decided["reference_label"]
).sum()

print(f"Individual annotation accuracy: {individual_accuracy:.1%}")
print(f"Majority accuracy: {correct / len(decided):.1%}")
print(f"Coverage: {len(decided) / len(majority):.1%}")
print(f"Correct labels across all items: {correct / len(majority):.1%}")
print(f"Tied items: {ties.sum()}")

### 4.2 Simulating Worker Qualification

Majority voting treats every annotation equally. But what if some workers are more reliable than others?

Crowdsourcing platforms can use **known-answer questions** to assess worker performance before or during annotation.

Let's simulate this process:

1. Take the first **5 annotations from each worker** as qualification questions.
2. Compare their responses with the reference labels.
3. Estimate each worker's reliability as the proportion of correct answers.
4. Reserve their remaining annotations for aggregation and evaluation.

For example, a worker who answers 4 of 5 qualification questions correctly receives a reliability score of **0.80**.

**Question:** How much can we trust a reliability estimate based on only five questions?

In [ ]:
# Reserve 5 annotations per worker for simulated qualification.
qualification = rte.groupby("worker_id", sort=False).head(5).copy()

# Keep the remaining annotations for aggregation.
evaluation = rte.drop(qualification.index).copy()

# Estimate each worker's accuracy on qualification items.
qualification["correct"] = (
    qualification["response"] == qualification["reference_label"]
)

worker_reliability = (
    qualification.groupby("worker_id")["correct"].mean()
)

print("Qualification annotations:", len(qualification))
print("Remaining annotations:", len(evaluation))

# How many workers received each reliability score?
display(
    worker_reliability.value_counts()
    .sort_index()
    .rename_axis("Reliability")
    .to_frame("Workers")
)

**Simulation limitation:** In a real qualification test, workers would typically answer the same predefined control questions before starting the task. Here, we use the first five recorded annotations from each worker, which may be different items and do not necessarily reflect the actual order in which they were completed. The resulting reliability scores are therefore only illustrative.

### 4.3 Should Every Vote Count Equally?

Now we can use the estimated worker reliability to aggregate the remaining annotations.

We will compare two approaches:

- **Majority voting:** Every worker contributes one vote.
- **Reliability-weighted voting:** Each vote is weighted by the worker's estimated reliability.

For example, three workers with reliability 0.6 voting *Entailed* contribute a total weight of 1.8. Two workers with reliability 1.0 voting *Not entailed* contribute a total weight of 2.0.

Majority voting would choose *Entailed*, while weighted voting would choose *Not entailed*.

Let's apply both strategies to the **same remaining annotations**.

**Question:** Can weighting change the selected label or resolve a tie?

In [ ]:
import numpy as np

# Attach each worker's estimated reliability.
evaluation["weight"] = evaluation["worker_id"].map(worker_reliability)

# Weight votes for Entailed (1).
evaluation["weighted_entailed"] = (
    evaluation["response"] * evaluation["weight"]
)

# Aggregate the remaining annotations by item.
comparison = evaluation.groupby("item_id").agg(
    total=("response", "size"),
    entailed=("response", "sum"),
    weighted_entailed=("weighted_entailed", "sum"),
    total_weight=("weight", "sum"),
    reference_label=("reference_label", "first")
)

# Ordinary majority voting.
comparison["majority_label"] = (
    comparison["entailed"] > comparison["total"] / 2
).astype("Int64")

ties = comparison["entailed"] == comparison["total"] / 2
comparison.loc[ties, "majority_label"] = pd.NA

# Reliability-weighted voting.
comparison["weighted_label"] = (
    comparison["weighted_entailed"] > comparison["total_weight"] / 2
).astype("Int64")

weighted_ties = np.isclose(
    comparison["weighted_entailed"],
    comparison["total_weight"] / 2
)
comparison.loc[weighted_ties, "weighted_label"] = pd.NA

In [ ]:
total_items = rte["item_id"].nunique()

# Compare accuracy and coverage for each method.
for label in ["majority_label", "weighted_label"]:
    decided = comparison.dropna(subset=[label])
    accuracy = (decided[label] == decided["reference_label"]).mean()
    coverage = len(decided) / total_items

    print(f"{label}:")
    print(f"  Accuracy: {accuracy:.1%}")
    print(f"  Coverage: {coverage:.1%}")

# Compare accuracy on items classified by both methods.
both = comparison.dropna(
    subset=["majority_label", "weighted_label"]
)

print("\nAccuracy on the same decided items:")
for label in ["majority_label", "weighted_label"]:
    accuracy = (both[label] == both["reference_label"]).mean()
    print(f"  {label}: {accuracy:.1%}")

# Find cases where the two methods select different labels.
changed = both[
    both["majority_label"] != both["weighted_label"]
]

print(f"\nDifferent decisions: {len(changed)}")
display(changed.head(5))

## 5. Quality Control Through Worker Selection

### 5.1 Should We Exclude Less Reliable Workers?

Previously, we estimated each worker's reliability using five simulated qualification questions.

Now let's use those scores to **select which workers can contribute to our final labels**.

For example, we can require workers to answer at least **80% of qualification questions correctly**.

We will:

1. Keep only workers who meet the qualification threshold.
2. Aggregate their remaining annotations using majority voting.
3. Compare the results with our previous aggregation strategies.

**Question:** Does selecting more reliable workers improve annotation quality? What might we lose by excluding other workers?

In [ ]:
# Select workers who passed qualification.
threshold = 0.6 # better than random
qualified = evaluation[evaluation["weight"] >= threshold].copy()

# Aggregate qualified annotations.
qualified["weighted_entailed"] = (
    qualified["response"] * qualified["weight"]
)

filtered = qualified.groupby("item_id").agg(
    total=("response", "size"),
    entailed=("response", "sum"),
    weighted_entailed=("weighted_entailed", "sum"),
    total_weight=("weight", "sum")
)

# Majority voting among qualified workers.
filtered["filtered_majority"] = (
    filtered["entailed"] > filtered["total"] / 2
).astype("Int64")

ties = filtered["entailed"] == filtered["total"] / 2
filtered.loc[ties, "filtered_majority"] = pd.NA

# Weighted voting among qualified workers.
filtered["filtered_weighted"] = (
    filtered["weighted_entailed"] > filtered["total_weight"] / 2
).astype("Int64")

ties = np.isclose(
    filtered["weighted_entailed"],
    filtered["total_weight"] / 2
)
filtered.loc[ties, "filtered_weighted"] = pd.NA

# Add both filtered strategies to the comparison.
# Update the filtered results (safe to rerun).
comparison = comparison.drop(
    columns=["filtered_majority", "filtered_weighted"],
    errors="ignore"
)

comparison = comparison.join(
    filtered[["filtered_majority", "filtered_weighted"]]
)

### 5.2 Comparing Quality-Control Strategies

We now have three ways of combining the annotations:

- **Majority voting:** Every remaining annotation counts equally.
- **Reliability-weighted voting:** Votes are weighted by qualification performance.
- **Worker filtering:** Only workers who passed qualification contribute.

Let's compare the accuracy and coverage of all three methods.

**Remember:** Accuracy is calculated on items receiving a label. Coverage is the proportion of all original items that receive a label.

In [ ]:
methods = {
    "All + majority": "majority_label",
    "All + weighted": "weighted_label",
    "Qualified + majority": "filtered_majority",
    "Qualified + weighted": "filtered_weighted",
}

results = []

for name, column in methods.items():
    decided = comparison.dropna(subset=[column])

    results.append({
        "Method": name,
        "Accuracy": (
            decided[column] == decided["reference_label"]
        ).mean(),
        "Coverage": len(decided) / rte["item_id"].nunique()
    })

display(
    pd.DataFrame(results)
    .set_index("Method")
    .style.format({
        "Accuracy": "{:.1%}",
        "Coverage": "{:.1%}"
    })
)

## 6. Continuous Quality Control

### 6.1 Monitoring Annotators with Control Items

We have used qualification questions to estimate worker reliability before aggregating their annotations.

Another approach is to **monitor annotation quality continuously** by inserting known-answer control questions into the annotation workflow.

Let's simulate this process using our RTE dataset:

1. Every **5th annotation** from each worker is treated as a control item.
2. Control responses are compared with their reference labels.
3. We calculate each worker's running accuracy on control items.
4. If accuracy falls below a threshold, the worker is deactivated and **all subsequent annotations are discarded**.

We will require at least three control questions before making a decision, so workers are not excluded after a single mistake.

Since the dataset does not contain the original annotation sequence, we will simulate one by shuffling the records.

**Question:** How much annotation work could continuous quality control prevent us from accepting?

In [ ]:
# Simulation settings
CONTROL_INTERVAL = 5  # Every 5th annotation from each worker is a control item.
MIN_CONTROLS = 3       # Minimum number of control items before a worker can be deactivated.
THRESHOLD = 0.7        # Minimum required accuracy on control items (70%).
                       # Workers falling below this threshold are deactivated.

# Simulate an annotation order (no sampling or data loss).
stream = rte.sample(frac=1, random_state=42).copy()

# Assign a sequence position within each worker's annotations.
stream["position"] = stream.groupby("worker_id").cumcount() + 1
stream["is_control"] = stream["position"] % CONTROL_INTERVAL == 0

kept_indices = []
removed_workers = []

# Process each worker's annotations in their simulated order.
for worker_id, rows in stream.groupby("worker_id"):

    controls_seen = 0
    controls_correct = 0

    for idx, row in rows.iterrows():

        if row["is_control"]:
            # Check the answer to a control question.
            controls_seen += 1
            controls_correct += int(
                row["response"] == row["reference_label"]
            )

            reliability = controls_correct / controls_seen

            # Deactivate the worker if reliability is too low.
            if (controls_seen >= MIN_CONTROLS
                    and reliability < THRESHOLD):
                removed_workers.append({
                    "worker_id": worker_id,
                    "position": row["position"],
                    "reliability": reliability
                })
                break

        else:
            # Accept this regular annotation.
            kept_indices.append(idx)

# Result: regular annotations accepted before deactivation.
monitored = stream.loc[kept_indices].copy()

# All regular annotations, without worker deactivation.
baseline = stream[~stream["is_control"]].copy()

removed = pd.DataFrame(removed_workers)

print(f"Workers deactivated: {len(removed)}")
print(f"Regular annotations without filtering: {len(baseline)}")
print(f"Regular annotations retained: {len(monitored)}")
print(f"Future annotations discarded: {len(baseline) - len(monitored)}")

print("Sample of removed workers: ")
display(removed.head())

### 6.2 Evaluating Continuous Quality Control

We have simulated a system that monitors each worker's performance and stops accepting future annotations when their control-item accuracy becomes too low.

Now let's evaluate the consequences.

Using majority voting in both cases, compare:

- **Accuracy:** How often do aggregated labels match the reference?
- **Coverage:** How many original items receive a majority label?
- **Annotation retention:** How many regular annotations remain available?

**Questions:**
- Does continuous quality control improve reference-based accuracy?
- What happens to annotation coverage?
- Could an incorrect decision to deactivate a worker make the final dataset worse?

In [ ]:
# Reference labels for all original items.
references = rte.groupby("item_id")["reference_label"].first()

def evaluate_majority(data):
    votes = data.groupby("item_id")["response"].agg(
        total="count",
        entailed="sum"
    )

    # Assign a label only if one response has a strict majority.
    labels = (votes["entailed"] > votes["total"] / 2).astype("Int64")

    ties = votes["entailed"] == votes["total"] / 2
    labels.loc[ties] = pd.NA

    # Include items with no remaining annotations as undecided.
    labels = labels.reindex(references.index)
    decided = labels.notna()

    return {
        "Annotations": len(data),
        "Decided items": decided.sum(),
        "Coverage": decided.mean(),
        "Accuracy": (
            labels[decided] == references[decided]
        ).mean()
    }

results = pd.DataFrame({
    "Without monitoring": evaluate_majority(baseline),
    "With monitoring": evaluate_majority(monitored)
}).T

display(
    results.style.format({
        "Annotations": "{:.0f}",
        "Decided items": "{:.0f}",
        "Coverage": "{:.1%}",
        "Accuracy": "{:.1%}"
    })
)


What is the limitation of this simulation?

## 7. When Disagreement Is Not Necessarily an Error

So far, we have examined disagreement in a classification task, where annotations can be compared against an accepted reference label.

But what happens when people are asked to make a **subjective judgment**?

The Emotion dataset (Snow et al., 2008) contains ratings of news headlines on different emotional dimensions.

We will focus on **valence**: how negative or positive a headline feels, on a scale from −100 to +100.

Let's compare how different people rated two headlines.

**Questions:**
- Do annotators agree equally strongly on both headlines?
- Why might the second headline produce more varied responses?
- Does greater disagreement necessarily mean lower annotation quality?

In [ ]:
# Load emotion ratings and select valence.
emotion = pd.read_csv("data/processed/emotion.csv")
valence = emotion[emotion["dimension"] == "valence"]

# Calculate the mean and standard deviation for each headline.
rating_stats = valence.groupby("item_id").agg(
    headline=("text", "first"),
    mean=("response", "mean"),
    std=("response", "std")
)

# Sort from lowest to highest variation.
rating_stats = rating_stats.sort_values("std")

# Headlines with the lowest variation.
display(rating_stats.head(10).round(1))

# Headlines with the highest variation.
display(rating_stats.tail(10).round(1))

In [ ]:
print("High agreement====: ")
print(rating_stats.iloc[0,])
print()
print("Low agreement====:")
print(rating_stats.iloc[-1,])

In [ ]:
import numpy as np

# Select items by position in the sorted dataframe.
low_id = rating_stats.index[0]     # Lowest variation
high_id = rating_stats.index[-1]   # Highest variation

# Show their headlines and statistics.
display(rating_stats.loc[[low_id, high_id]].round(1))

# Plot individual ratings.
fig, ax = plt.subplots(figsize=(9, 3.5))
rng = np.random.default_rng(42)

for y, item_id in enumerate([low_id, high_id]):
    ratings = valence.loc[
        valence["item_id"] == item_id, "response"
    ]

    # Small jitter to make overlapping ratings visible.
    ax.scatter(
        ratings,
        y + rng.uniform(-0.1, 0.1, len(ratings)),
        alpha=0.7
    )

    # Mark the mean.
    ax.scatter(ratings.mean(), y, marker="D", s=100, color="black")

ax.set_yticks([0, 1], ["Low variation", "High variation"])
ax.set_xlim(-105, 105)
ax.set_xlabel("Valence rating (−100 to +100)")
ax.set_title("Variation in human judgments")
plt.tight_layout()
plt.show()

**Why do annotators disagree?**

Look at the headlines with the greatest variation in valence ratings.

- Can you explain why people might rate the same headline differently?
- Could disagreement result from different perspectives, ambiguous instructions, or both?
- What additional information would help distinguish between these explanations?

**Important:** High variation tells us that judgments differ - not why. Before treating disagreement as an error, we need to understand the task and the people providing the judgments.

## 7. Applying Annotation Quality to your Project

Conduct a small annotation study using **20–30 items** from your project dataset, annotated independently by **3–5 people**.

Use Label Studio to collect the annotations. Help each other in order to collect enough judgements. 

Then, using Python:

1. **Evaluate annotation quality.** Calculate suitable agreement or disagreement measures, compare annotations with the dataset's reference labels, and inspect problematic items.
2. **Simulate one quality-control mechanism.** For example, use a subset of items as known-answer controls to estimate worker reliability, then simulate worker filtering or reliability-weighted aggregation.
3. **Evaluate the effect.** Compare the baseline with your simulated quality-control strategy using reference-based accuracy, annotation coverage, and any other relevant measures.
4. **Document and reflect.** Explain your annotation procedure, quality-control rules, results, and limitations.

**Main question:** Did the quality-control mechanism improve the resulting annotations? What trade-offs or limitations did you observe?

You do not need to implement quality-control mechanisms directly in Label Studio. They can be simulated after collecting the annotations.